# Nairobi OS Quickstart: The Sovereign API

Welcome to the **Heavy Iron**. This notebook demonstrates the v0.3.1 "Frictionless" API refit of Nairobi OS.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chege/nairobi-connector-open-source/blob/main/quickstart.ipynb)

Nairobi OS is a distributed microservice architecture designed for high-performance, zero-copy data analysis. By offloading heavy lifting to a specialized Rust-based refinery daemon and using `memfd` handles, we achieve performance that makes Pandas look like a toy.

## 🚀 Why Nairobi OS?

- **Zero-Copy Ingestion**: Data stays in kernel memory (`memfd`).
- **Rust-Powered Analytics**: All statistical operations are executed in parallelized Rust.
- **Lagos Visual Cortex**: Hardware-accelerated plotting directly from shared memory.

### 🛠️ Google Colab Setup (Run this first if on Colab)

Nairobi OS requires a D-Bus session and the Rust binaries to be built. The following cell handles the environment setup for headless Colab instances.

In [ ]:
import sys
import os
import subprocess
import shutil
import time
import glob

def log_step(msg):
    print(f"[FORENSIC LOG {time.strftime('%H:%M:%S')}] {msg}")

if 'google.colab' in sys.modules:
    log_step("STARTING EXTREME VERBOSE INITIALIZATION")
    log_step("Environment: Google Colab Managed Runtime")

    # 1. System Dependencies
    log_step("Updating apt-get package indices...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)

    deps = ["dbus-x11", "build-essential", "curl", "pkg-config", "libssl-dev", "clang", "llvm-dev", "libclang-dev"]
    log_step(f"Injecting low-level system dependencies: {', '.join(deps)}")
    subprocess.run(["apt-get", "install", "-y", "-qq"] + deps, check=True)

    os.environ["LIBCLANG_PATH"] = "/usr/lib/llvm-14/lib"

    # 2. Python & Rust Tooling
    log_step("Installing Maturin (Python/Rust bridge tool)...")
    subprocess.run(["pip", "install", "maturin"], check=True)

    cargo_path = shutil.which("cargo")
    if cargo_path is None:
        log_step("CRITICAL: Rust toolchain not found. Bootstrapping...")
        subprocess.run("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y", shell=True, check=True)
        os.environ["PATH"] += os.pathsep + os.path.expanduser("~/.cargo/bin")
    else:
        log_step(f"Rust toolchain verified at: {cargo_path}")

    # 3. D-Bus IPC Layer
    log_step("Initializing D-Bus session bus...")
    try:
        dbus_output = subprocess.check_output(["dbus-launch"]).decode()
        for line in dbus_output.splitlines():
            if "=" in line:
                key, value = line.split("=", 1)
                os.environ[key] = value
        log_step("D-Bus session environment variables exported.")
    except Exception as e:
        log_step(f"WARNING: D-Bus launch error: {e}")

    # 4. Repository Acquisition
    repo_dir = "/content/nairobi-connector-open-source"
    if not os.path.exists(repo_dir):
        log_step("Cloning Nairobi OS source...")
        subprocess.run(["git", "clone", "https://github.com/KevinKenya/nairobi-connector-open-source.git", repo_dir], check=True)
    else:
        log_step("Syncing repository...")
        subprocess.run(["git", "-C", repo_dir, "pull"], check=True)

    os.chdir(repo_dir)

    # 5. Compilation
    log_step("--- COMMENCING HEAVY IRON FORGE (BUILD_WHEEL.SH) ---")
    build_script = os.path.join(repo_dir, "build_wheel.sh")
    subprocess.run(["chmod", "+x", build_script], check=True)

    process = subprocess.Popen(["bash", "-l", build_script], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(f"[BUILD]: {line.strip()}")
    process.wait()

    if process.returncode != 0:
        log_step(f"FAILURE: Build script exited with code {process.returncode}")
        sys.exit(1)

    # 6. Installation (Using glob to find the forged wheel in target/wheels)
    wheel_pattern = os.path.join(repo_dir, "target/wheels/*.whl")
    wheels = glob.glob(wheel_pattern)

    if wheels:
        latest_wheel = max(wheels, key=os.path.getctime)
        log_step(f"Detected forged artifact: {latest_wheel}")
        subprocess.run(["pip", "install", "--force-reinstall", latest_wheel], check=True)
        log_step("✅ TOTAL INITIALIZATION SUCCESSFUL. REFINERY READY.")
    else:
        log_step("ERROR: Build script reported success but no .whl found in target/wheels!")
        sys.exit(1)
else:
    log_step("Native Linux environment detected. Skipping managed setup steps.")

[FORENSIC LOG 22:10:12] STARTING EXTREME VERBOSE INITIALIZATION
[FORENSIC LOG 22:10:12] Environment: Google Colab Managed Runtime
[FORENSIC LOG 22:10:12] Updating apt-get package indices...
[FORENSIC LOG 22:10:22] Injecting low-level system dependencies: dbus-x11, build-essential, curl, pkg-config, libssl-dev, clang, llvm-dev, libclang-dev
[FORENSIC LOG 22:10:51] Installing Maturin (Python/Rust bridge tool)...


### 1. The Ignition

In Nairobi OS, we don't just 'import' data; we ignite the refinery. The `connect()` function (a semantic alias for `start_refinery`) ensures the D-Bus session is active and the Axum Refinery daemon is ready for action.

In [ ]:
import nairobi_os as nb
import kagglehub
import os

# Ignite the Refinery
nb.connect()

### 2. The Ingestion

We'll use the NBA Player Statistics dataset. `nb.read_csv()` returns a `SovereignFrame`, which is a high-level wrapper around a Rust `memfd` handle.

In [ ]:
# Download NBA dataset
dataset_path = kagglehub.dataset_download('vivovinco/nba-player-stats')
csv_file = os.path.join(dataset_path, '2021-2022 NBA Player Stats - Regular.csv')

# Ingest into the Sovereign Frame
df = nb.read_csv(csv_file)
print(f"Sovereign Handle ID: {df.handle_id}")

### 3. The Forensic Audit (`.mean()`, `.std_dev()`...)

Nairobi OS provides a fluent column-accessor API (`df.PTS.mean()`). A full statistical analysis is computed efficiently in Rust on the first call, and all subsequent metrics (like max, std_dev) use the cached result with zero measurable latency.

In [ ]:
# Analyze points per game
mean_pts = df.PTS.mean()
std_pts = df.PTS.std_dev()
max_pts = df.PTS.max()

print("--- NBA Points Statistics ---")
print(f"Mean: {mean_pts:.4f}")
print(f"Std Dev: {std_pts:.4f}")
print(f"Max: {max_pts:.4f}")

### 4. The Relational Strike (`.correlate()`)

Calculating correlation matrices for large datasets can be slow in Python. Nairobi OS executes this on the metal.

In [ ]:
# Correlate Points, Assists, and Rebounds
corr_matrix = df.correlate("PTS,AST,TRB")

print("--- Correlation Matrix ---")
import json
print(json.dumps(corr_matrix, indent=2))

### 5. The Distillation (`.query()`)

Sometimes you only need a subset of the iron. `.query()` executes SQL directly on the `memfd` and returns a **new** `SovereignFrame` containing only the distilled data.

In [ ]:
# Extract only the points for high-performance visualization
distilled_df = df.query("SELECT PTS FROM dataset")
print(f"New Sovereign Handle: {distilled_df.handle_id}")

### 6. The Visual Cortex (`.plot()`)

Lagos Vision maps the `memfd` directly into the GPU pipeline. No data ever passes through the Python interpreter during rendering.

In [ ]:
# Render the distilled points
distilled_df.plot(width=800, height=400)

### 7. Shutdown

Clean up the refinery when you're done.

In [ ]:
nb.stop_refinery()